In [128]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN


In [129]:
symbols = pd.read_csv("symbols.csv")['Symbol'].tolist()
netAssets = pd.read_csv("net_assets.csv")
closes = pd.read_csv("closes.csv", index_col=0)

netAssets['Inception Date'] = pd.to_datetime(netAssets['Inception Date'])
netAssets = netAssets[netAssets['Inception Date'] <= pd.Timestamp('2011-01-01')]
goodSymbols = netAssets['Symbol'].tolist()
goodCloses = closes[goodSymbols]
goodCloses = goodCloses.loc['2010-09-22':]

**I need to find clusters for hyperparameter tuning for like ETFs. If I do it on a per ETF instance, it becomes way to susceptible to overfitting, but if I do it on a generalized basis, it doesn't allow for good convergence. I am choosing DBSCAN because it can handle noise and I don't know how many clusters there will be.**

In [130]:
returns = goodCloses.pct_change().dropna()
corr = returns.corr()
dist = np.sqrt(2 * (1 - corr))



In [ ]:
#DBSCAN Clustering
DBmodel = DBSCAN(eps=0.5, min_samples=2, metric='precomputed')
DBclusters = DBmodel.fit_predict(dist)

uniqueClusters = set(DBclusters)
for cluster in uniqueClusters:
    if cluster == -1:
        print("No fit:")
    else:
        print(f"Cluster {cluster + 1}:")
    clusterSymbols = [goodSymbols[j] for j in range(len(goodSymbols)) if DBclusters[j] == cluster]
    print(f"{', '.join(clusterSymbols)}")


**I like that the DBSCAN algorithm gave me two strong clusters. The third is ehh whatever, but interesting in the fact that the ETFs in cluster 3 don't cluster well with those in 'Noise' because it is gold based. With that, I'm going to manually make the three clusters as this:

Cluster 1: [VONG, VOO, VOOG, TQQQ, SCHG, SCHF, SCHX, SCHB, VT, ACWI, MGK, DFAC, VEA, SPDW, VEU, VYM, VIG, EFV, VWO, VGK, VV, VTV, VO, VBR, VUG, VGT, VB, ITOT, RSP, EEM, VXF, EFA, IWR, SOXX, VTI, SPYG, SPYV, IUSG, IJR, IWF, IWM, IVE, IWD, IVW, IJH, IVV, IWB, QQQ, XLK, XLI, XLF, DIA, MDY, SPY, **XLV**, **XLE**]

Cluster 2: [VCIT, VGIT, VCSH, VGSH, BND, BSV, BIV, MBB, AGG, LQD, TLT, IEF, **MUB**, **BIL**, ]

Cluster 3: [IAU, GLD, **GDX**, **SLV**]


Note MUB and BIL made it into cluster 2 because this is entirely comprised of fixed income ETFs

GDX and SLV made it into cluster 3 becaus these comprise of Gold and Silver ETFs

XLV and XLE made it into cluster 1 because they don't belong in 1 or 2, but additonally, other subsector ETFs are in cluster 1


In [132]:
def stochastic(df, n=14):
    low = df['Close'].rolling(window=n).min()
    high = df['Close'].rolling(window=n).max()

    dnom = high - low
    dnom = dnom.replace(0, 1e-9)

    k = (df['Close'] - low) / dnom * 100
    d = k.rolling(window=3).mean()

    return k, d

def calcRegimeFilter(df):
    df = df.copy()

    df['Bull Regime'] = df['Close'] > df['SMA_200']
    df['Bear Regime'] = df['Close'] < df['SMA_200']

    df['High Volatility Regime'] = df['vol_20'] > df['vol_50']
    df['Low Volatility Regime'] = df['vol_20'] < df['vol_50']

    df['Regime'] = np.select(
        [
            (df['Bull Regime'] & df['Low Volatility Regime']),
            (df['Bull Regime'] & df['High Volatility Regime']),
            (df['Bear Regime'] & df['Low Volatility Regime']),
            (df['Bear Regime'] & df['High Volatility Regime'])
        ],
        [
            'Bull-Low Vol',
            'Bull-High Vol',
            'Bear-Low Vol',
            'Bear-High Vol'
        ],
        default='No Regime'
    )

    

    return df

def getTarget(df):
    df = df.copy()
    df['Target'] = np.where(df['Returns'].shift(-1) > 0, 1, 0)
    return df

def getTechnicalData(closes):
    df = pd.DataFrame()
    df['Close'] = closes
    df['Returns'] = closes.pct_change()
    df['SMA_20'] = closes.rolling(window=20).mean()
    df['SMA_50'] = closes.rolling(window=50).mean()
    df['SMA_200'] = closes.rolling(window=200).mean()

    df['vol_20'] = closes.rolling(window=20).std()
    df['vol_50'] = closes.rolling(window=50).std()

    df = calcRegimeFilter(df)
    df['K'], df['D'] = stochastic(df)
    df['K_minus_D'] = df['K'] - df['D']
    df['K_cross_D'] = np.where((df['K'] > df['D']) & (df['K'].shift(1) <= df['D'].shift(1)), 1, 0)
    df['dist_ma'] = (df['Close'] - df['SMA_200']) / df['SMA_200']
    df['vol_ratio'] = df['vol_20'] / df['vol_50']
    df['Target'] = np.where(df['Returns'].shift(-1) > 0, 1, 0)

    return df.dropna()






In [ ]:
#Test for the big loop

spyTest = goodCloses['SPY']
technicalSPY = getTechnicalData(spyTest)
dummyVars = pd.get_dummies(technicalSPY['Regime']).astype(int)
dummyVars['Standardized Regime'] = np.where(
    technicalSPY['Regime'] == 'Bull-Low Vol', 0, np.where(
        technicalSPY['Regime'] == 'Bull-High Vol', 1, np.where(
            technicalSPY['Regime'] == 'Bear-Low Vol', 2, np.where(
                technicalSPY['Regime'] == 'Bear-High Vol', 3, 4
            )
        )
    )
)
technicalSPY = pd.concat([technicalSPY, dummyVars['Standardized Regime']], axis=1)
technicalSPY




In [ ]:
goodCloses

In [135]:
technicalData = {}

for symbol in goodSymbols:
    technicalDF = getTechnicalData(goodCloses[symbol])
    dummyVars = pd.get_dummies(technicalDF['Regime']).astype(int)
    dummyVars['Standardized Regime'] = np.where(
        technicalSPY['Regime'] == 'Bull-Low Vol', 0, np.where(
            technicalSPY['Regime'] == 'Bull-High Vol', 1, np.where(
                technicalSPY['Regime'] == 'Bear-Low Vol', 2, np.where(
                    technicalSPY['Regime'] == 'Bear-High Vol', 3, 4
                    )
                )
            )
    )
    technicalDF = pd.concat([technicalDF, dummyVars['Standardized Regime']], axis=1)
    technicalData[symbol] = technicalDF




In [136]:
clusterOne = ['VONG', 'VOO', 'VOOG', 'TQQQ', 'SCHG', 'SCHF', 'SCHX', 'SCHB', 'VT', 'ACWI', 'MGK', 'DFAC', 'VEA', 'SPDW', 'VEU', 'VYM', 'VIG', 'EFV', 'VWO', 'VGK', 'VV', 'VTV', 'VO', 'VBR', 'VUG', 'VGT', 'VB', 'ITOT', 'RSP', 'EEM', 'VXF', 'EFA', 'IWR', 'SOXX', 'VTI', 'SPYG', 'SPYV', 'IUSG', 'IJR', 'IWF', 'IWM', 'IVE', 'IWD', 'IVW', 'IJH', 'IVV', 'IWB', 'QQQ', 'XLK', 'XLI', 'XLF', 'DIA', 'MDY', 'SPY', 'XLV', 'XLE']
clusterTwo = ['VCIT', 'VGIT', 'VCSH', 'VGSH', 'BND', 'BSV', 'BIV', 'MBB', 'AGG', 'LQD', 'TLT', 'IEF', 'MUB', 'BIL']
clusterThree = ['IAU', 'GLD', 'GDX', 'SLV']

In [ ]:
trainedModels = {}
logisticRegressionFeatures = [] 
features = ['K', 'D', 'K_minus_D', 'K_cross_D', 'dist_ma', 'vol_ratio', 'Standardized Regime'] 

XTrainDict = {}
XTestDict = {}
YTrainDict = {}
YTestDict = {}


for symbol in goodSymbols:
    df = technicalData[symbol].copy()
    X = df[features]
    y = df['Target']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    XTrainDict[symbol] = X_train_scaled
    XTestDict[symbol] = X_test_scaled
    YTrainDict[symbol] = y_train
    YTestDict[symbol] = y_test

    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)

    trainedModels[symbol] = model

    print(f'Train Accuracy for {symbol}: {model.score(X_train_scaled, y_train):.4f},  Test Accuracy for {symbol}: {model.score(X_test_scaled, y_test):.4f}')
    



In [138]:
def getSharpeRatio(returns, risk_free_rate=0.0,periods_per_year=252):
    returns = pd.Series(returns).dropna()
    if len(returns) == 0:

        return np.nan
    excess_returns = returns - (risk_free_rate / periods_per_year)
    volatility = excess_returns.std(ddof=1)
    if volatility == 0:
        return np.nan
    
    annualized_excess_return = excess_returns.mean() * periods_per_year
    annualized_volatility = volatility * np.sqrt(periods_per_year)

    return annualized_excess_return / annualized_volatility

def getTrades(signaledProbabilities):
    probabilities = pd.DataFrame(signaledProbabilities)
    probabilities.set_index(goodCloses.index[-745:])

    signaledTrades = probabilities.where((probabilities > 0.55) | (probabilities < 0.45))

    longTrades = signaledTrades.where(signaledTrades > 0.5)
    longTrades.index = goodCloses.index[-745:]
    longTrades = longTrades[:-1]

    shortTrades = signaledTrades.where(signaledTrades < 0.5)
    shortTrades.index = goodCloses.index[-745:]
    shortTrades = shortTrades[:-1]

    returns = closes.pct_change().shift(-1)
    returns = returns.loc[shortTrades.index[0]:]
    shortReturns = -1 * returns.where(shortTrades > 0)
    longReturns = returns.where(longTrades > 0)
    combinedReturns = longReturns.fillna(0) + shortReturns.fillna(0)


    trades = (combinedReturns != 0).sum(axis=1)
    return combinedReturns, trades

In [ ]:
LRprobabilities = {}
for symbol in goodSymbols:
    LRprobabilities[symbol] = trainedModels[symbol].predict_proba(XTestDict[symbol])[:, 1]

LRTrades, LRTradesAGG = getTrades(LRprobabilities)

dailyReturns = LRTrades.sum(axis=1) / LRTradesAGG
dailyReturns = dailyReturns.fillna(0)

sharpeRatio = getSharpeRatio(dailyReturns.values)
sharpeRatio


In [140]:
sp500 = pd.read_csv('S&P500.csv')
sp500 = sp500.set_index('Date')
sp500Return = sp500.pct_change().dropna()

spWealthIndex = (sp500Return['^GSPC']+1).cumprod()
spSharpeRatio = getSharpeRatio(sp500Return['^GSPC'], periods_per_year=252)


In [ ]:
wealthIndex = (dailyReturns+1).cumprod()
wealthIndex.plot()

In [ ]:
print(f'''This strategy yielded a {100*(wealthIndex.iloc[-1] - wealthIndex.iloc[0]/wealthIndex.iloc[0])}% return\n
      the S&P 500 had yieled a {100*(spWealthIndex.iloc[-1] - spWealthIndex.iloc[0]/spWealthIndex.iloc[0])}% return''')

In [143]:
def plot_drawdown(returns, title="Strategy Drawdown", starting_value=1):
    rets = pd.Series(returns).dropna()

    wealth_index = starting_value * (1 + rets).cumprod()
    previous_peaks = wealth_index.cummax()
    drawdowns = (wealth_index - previous_peaks) / previous_peaks

    max_drawdown = drawdowns.min()
    max_drawdown_date = drawdowns.idxmin()

    fig, ax1 = plt.subplots(figsize=(12, 6))

    ax1.plot(wealth_index, label="Wealth Index")
    ax1.plot(previous_peaks, label="Previous Peaks")
    ax1.set_ylabel("Wealth Index")
    ax1.set_title(title)

    ax2 = ax1.twinx()
    ax2.plot(drawdowns, label="Drawdown", alpha=0.6)
    ax2.set_ylabel("Drawdown")

    ax2.annotate(
        f"Max Drawdown: {max_drawdown:.2%}",
        xy=(max_drawdown_date, max_drawdown),
        xytext=(max_drawdown_date, max_drawdown * 0.5),
        arrowprops=dict(arrowstyle="->", lw=1),
    )

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right")

    plt.show()

    return {
        "wealth_index": wealth_index,
        "previous_peaks": previous_peaks,
        "drawdowns": drawdowns,
        "max_drawdown": max_drawdown,
        "max_drawdown_date": max_drawdown_date
    }

In [ ]:
plot_drawdown(sp500Return['^GSPC'])

In [ ]:
plot_drawdown(dailyReturns)

In [146]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

In [ ]:
XGBOOSTXTrainDict = {}
XGBOOSTXTestDict = {}
XGBOOSTYTrainDict = {}
XGBOOSTYTestDict = {}
XGModels = {}
XGProbabilities = {}

for symbol in goodSymbols:
    df = technicalData[symbol].copy()
    X = df[features]
    y = df['Target']

    XTrain, XTest, yTrain, yTest = train_test_split(X, y, test_size=0.2, shuffle=False)

    scaler = StandardScaler()
    

    XGBOOSTXTrainDict[symbol] = X_train
    XGBOOSTXTestDict[symbol] = X_test
    XGBOOSTYTrainDict[symbol] = y_train
    XGBOOSTYTestDict[symbol] = y_test

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss"
        )

    model.fit(XTrain, yTrain)

    XGModels[symbol] = model

    XGProbabilities[symbol] = pd.Series(
        model.predict_proba(XTest)[:, 1]
    )
    print(f'Train Accuracy for {symbol}: {model.score(XTrain, yTrain):.4f},  Test Accuracy for {symbol}: {model.score(XTest, yTest):.4f}')


    

In [ ]:
XGTrades, XGTradesAGG = getTrades(XGProbabilities)

XGdailyReturns = XGTrades.sum(axis=1) / XGTradesAGG
XGdailyReturns = XGdailyReturns.fillna(0)

XGsharpeRatio = getSharpeRatio(XGdailyReturns.values)
XGsharpeRatio


In [ ]:
plot_drawdown(XGdailyReturns)

In [150]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [151]:
def makeSequences(X, y, lookback=20):
    XSeq = []
    ySeq = []

    for i in range(lookback, len(X)):
        XSeq.append(X[i-lookback:i])
        ySeq.append(y[i])

    return np.array(XSeq), np.array(ySeq)

In [ ]:
LTSMfeatureCollection = {}

for symbol in goodSymbols:

    df = technicalData[symbol].copy()
    df['LRProbability'] = XGProbabilities[symbol]
    df['XGProbability'] = LRprobabilities[symbol]

    

    X = df[features]
    y = df['Target']

    train_test_split(X, y, test_size=0.2, shuffle=False)



    
